In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer.BufferTrainer import BufferTrainer
from src.data_utils import (
    get_mnist_tasks,
    _extract_targets,
    get_context_sets,
    create_holdout_set,
)
from src.utils.general import InContextHead, print_colored
from src import models
from src.buffer import MultiTaskBuffer

from src.regulariser import UnbiasRegulariser, L2Regulariser, MultiRegulariser

from configs import MNIST_BUFFER_CONFIG as CONFIG

### Task Incremental Training

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(head=head, device="cuda", seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
buffer = MultiTaskBuffer([])
buffer_trainer = BufferTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    buffer=buffer,
    paradigm="TIL",
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    buffer_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
        context_id=i,
        target_acc=0.9,
    )
    buffer_trainer.test(test_tasks, context_list=list(range(len(test_tasks))))

    if i < len(test_tasks) - 1:
        buffer_trainer.compute_rashomon_set(test, context_id=i)

### Domain Incremental Training

In [ ]:
SEED = 0
train_tasks, buffer_tasks, test_tasks = get_mnist_tasks(
    seed=SEED, train_val_split_ratio=0.95
)

model = models.get_mnist_model(device="cuda", output_dim=2, seed=SEED)

print([len(buffer) for buffer in buffer_tasks])

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [ ]:
MAX_BUFFER_TASKS = 2

buffer = MultiTaskBuffer([])
buffer_trainer = BufferTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="DIL",
    seed=SEED,
    buffer=buffer,
    domain_map_fn=domain_map_fn,
)

use_outer_bbox = False
for i, (train, buffer, test) in enumerate(zip(train_tasks, buffer_tasks, test_tasks)):
    buffer_trainer.train(
        train,
        test,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
        target_acc=0.8,
        use_outer_bbox=use_outer_bbox,
    )
    buffer_trainer.test(test_tasks)
    if i < len(train_tasks) - 1:
        buffer_trainer.compute_rashomon_set(test)
        if buffer_trainer.buffer.num_of_tasks() >= MAX_BUFFER_TASKS:
            buffer_trainer.remove_oldest_buffer()
            use_outer_bbox = True
        buffer_trainer.add_to_buffer(buffer)

### Class Incremental Learning

In [ ]:
SEED = 0
train_tasks, _, test_tasks = get_mnist_tasks(seed=SEED, train_val_split_ratio=1)

model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
train_tasks, buffer_tasks = zip(
    *[create_holdout_set(dataset) for dataset in train_tasks]
)
print([len(task) for task in buffer_tasks])

In [ ]:
MAX_BUFFER_TASKS = CONFIG["buffer_tasks"]

buffer = MultiTaskBuffer([])
buffer_trainer = BufferTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="CIL",
    seed=SEED,
    buffer=buffer,
)

use_outer_bbox = False
for i, (train, buffer, test) in enumerate(zip(train_tasks, buffer_tasks, test_tasks)):
    buffer_trainer.train(
        train,
        test,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
        target_acc=0.8,
        use_outer_bbox=use_outer_bbox,
    )
    buffer_trainer.test(test_tasks)
    if i < len(train_tasks) - 1:
        buffer_trainer.compute_rashomon_set(test)
        if buffer_trainer.buffer.num_of_tasks() >= MAX_BUFFER_TASKS:
            buffer_trainer.remove_oldest_buffer()
            use_outer_bbox = True
        buffer_trainer.add_to_buffer(buffer)

### Class Incremental Learning + Regulariser

In [ ]:
SEED = 0
train_tasks, _, test_tasks = get_mnist_tasks(seed=SEED, emnist=True, train_val_split_ratio=1)

model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
sizes = [4800, 4000, 4000, 3200, 0]
train_tasks, buffer_tasks = zip(
    *[create_holdout_set(dataset, holdout_size=holdout) for dataset, holdout in zip(train_tasks, sizes)]
)
print([len(task) for task in buffer_tasks])
print([len(task) for task in train_tasks])

In [ ]:
unbias = UnbiasRegulariser(
    lmbd=CONFIG["unbias_lambda"],
    unbias_domain=[
        torch.zeros(1, 1, 28, 28, device="cuda"),
        torch.ones(1, 1, 28, 28, device="cuda"),
    ],
)
l2 = L2Regulariser(lmbd=CONFIG["l2_lambda"])
regulariser = MultiRegulariser([l2, unbias])

In [ ]:
buffer = MultiTaskBuffer([])
buffer_trainer = BufferTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["initial_target_acc"],
    min_acc_increment=0,
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="CIL",
    seed=SEED,
    buffer=buffer,
)

MAX_BUFFER_CALLS = CONFIG["max_buffer_calls"]
target_acc = CONFIG["target_acc"]
lower_bounds = []
buffer_calls = []
for i, (train, test, buffer) in enumerate(zip(train_tasks, test_tasks, buffer_tasks)):
    buffer_trainer.train(
        train,
        test,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
        regulariser=regulariser,
    )
    results = buffer_trainer.test(test_tasks)
    accs = [res[1] for res in results]

    # If results are not satisfactory, then use buffer data to recompute rashomon set and continue training
    j = 0
    buffer_call = 0
    prev_acc = None
    while (
        j < MAX_BUFFER_CALLS
        and results[i][1] < target_acc
        and i > 0
        and not buffer_trainer.buffer.is_empty()
    ):
        buffer_call += 1
        print_colored("Using buffer to recompute LID.", color="amber")

        buffer_trainer.recall_dataset, (buffer_X, buffer_y) = buffer_trainer.buffer.consume_merge()
        print("Recall dataset size:", len(buffer_trainer.recall_dataset))
        dataset = torch.utils.data.TensorDataset(buffer_X, buffer_y)
        buffer_trainer.compute_rashomon_set(
            dataset, use_outer_bbox=False, batch_size=len(dataset)
        )
        buffer_trainer.train(
            train,
            test,
            batch_size=CONFIG["batch_size"],
            epochs=CONFIG["epochs"],
            lr=CONFIG["lr"],
            weight_decay=CONFIG["weight_decay"],
            regulariser=regulariser,
            early_stopping=True,
            val_freq=50,
            patience=10
        )
        results = buffer_trainer.test(test_tasks)
        accs = [res[1] for res in results]

        print("lower_bounds:", lower_bounds)
        diffs = [accs[i] - lower_bounds[i] for i in range(len(lower_bounds))]
        min_diff_idx = diffs.index(
            min(diffs)
        )  # The assumption is that the task closest to its boundary is the one restricting further improvements
        if results[i][1] > target_acc:
            break
        elif prev_acc is not None and results[i][1] - prev_acc < CONFIG["loosening_thresh"]:
            print("Loosening task", min_diff_idx, "bounds.")
            lower_bounds[min_diff_idx] = max(lower_bounds[min_diff_idx] - CONFIG["loosening_step"], 0.01)
            buffer_trainer.min_acc_limit = lower_bounds
        prev_acc = results[i][1]
        j += 1
    buffer_calls.append(buffer_call)

    print_colored(accs, color="green")

    lower_bounds.append(max(accs[i] - CONFIG["min_acc_increment"], 0.01))

    buffer_trainer.min_acc_limit = lower_bounds

    if i < len(train_tasks) - 1:
        buffer_trainer.compute_rashomon_set(test)
        buffer_trainer.add_to_buffer(buffer, task_id=i, k=CONFIG["buffer_k"])

    else:
        print("Buffer calls:", buffer_calls)


## Ablation Studies
### Large Buffer
#### Class Incremental Learning

In [4]:
import wandb

In [7]:
SMALL = 1000
MEDIUM = 5000
LARGE = 15000

def run_buffer(buffer_size: int, seed: int, config: wandb.config, paradigm="TIL"):
    def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
        """Map the global label to the in context label."""
        return labels % 2
    SEED = seed
    CONFIG = config
    train_tasks, _, test_tasks = get_mnist_tasks(seed=SEED, emnist=True, train_val_split_ratio=1)
    
    context_sets = get_context_sets(test_tasks)
    head = InContextHead(context_sets, 10, device="cuda")
    head.set_context(0)
    model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED, head = head if paradigm=="TIL" else None)
    print(
        f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
    )

    unbias = UnbiasRegulariser(
    lmbd=CONFIG["unbias_lambda"],
    unbias_domain=[
            torch.zeros(1, 1, 28, 28, device="cuda"),
            torch.ones(1, 1, 28, 28, device="cuda"),
        ],
    )
    l2 = L2Regulariser(lmbd=CONFIG["l2_lambda"])
    regulariser = MultiRegulariser([l2, unbias])

    if buffer_size == SMALL:
        sizes = [400, 400, 200, 0, 0]
    elif buffer_size == MEDIUM:
        sizes = [1400, 1200, 800, 600, 0]
    elif buffer_size == LARGE:
        sizes = [4800, 4000, 4000, 3200, 0]
    train_tasks, buffer_tasks = zip(
        *[create_holdout_set(dataset, holdout_size=holdout) for dataset, holdout in zip(train_tasks, sizes)]
    )
    print([len(task) for task in buffer_tasks])
    print([len(task) for task in train_tasks])

    task_labels = [torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]

    buffer = MultiTaskBuffer([])
    buffer_trainer = BufferTrainer(
        model,
        checkpoint=CONFIG["checkpoint"],
        n_iters=CONFIG["n_iters"],
        min_acc_limit=CONFIG["initial_target_acc"],
        min_acc_increment=0,
        primal_learning_rate=0.5 or CONFIG["primal_learning_rate"],
        dual_learning_rate=0.0001 or CONFIG["dual_learning_rate"],
        projection_strategy=CONFIG["projection_strategy"],
        n_certificate_samples=CONFIG["n_certificate_samples"],
        penalty_coefficient=CONFIG["penalty_coefficient"],
        paradigm=paradigm,
        seed=SEED,
        buffer=buffer,
        domain_map_fn=domain_map_fn if paradigm == "DIL" else None,
        task_labels = task_labels
    )

    if buffer_size == SMALL:
        MAX_BUFFER_CALLS = 1
    if buffer_size == MEDIUM:
        MAX_BUFFER_CALLS = 3
    if buffer_size == LARGE:
        MAX_BUFFER_CALLS = 7
    target_acc = CONFIG["target_acc"]
    lower_bounds = []
    buffer_calls = []
    accuracy_matrix = []
    for i, (train, test, buffer) in enumerate(zip(train_tasks, test_tasks, buffer_tasks)):
        buffer_trainer.train(
            train,
            test,
            batch_size=CONFIG["batch_size"],
            epochs=CONFIG["epochs"],
            lr=CONFIG["lr"],
            weight_decay=CONFIG["weight_decay"],
            regulariser=regulariser,
            context_id=i if paradigm == "TIL" else None
        )
        results = buffer_trainer.test(test_tasks, context_list=list(range(len(test_tasks))) if paradigm=="TIL" else [None] * len(test_tasks))
        accs = [res[1] for res in results]
        if i == 0 and accs[0] < 0.8:
            wandb.finish(1)
            return
        # If results are not satisfactory, then use buffer data to recompute rashomon set and continue training
        j = 0
        buffer_call = 0
        prev_acc = None
        while (
            j < MAX_BUFFER_CALLS
            and results[i][1] < target_acc
            and i > 0
            and not buffer_trainer.buffer.is_empty()
        ):
            buffer_call += 1
            print_colored("Using buffer to recompute LID.", color="amber")

            buffer_trainer.recall_dataset, (buffer_X, buffer_y) = buffer_trainer.buffer.consume_merge()
            print("Recall dataset size:", len(buffer_trainer.recall_dataset))
            dataset = torch.utils.data.TensorDataset(buffer_X, buffer_y)
            buffer_trainer.compute_rashomon_set(
                dataset, use_outer_bbox=False, batch_size=len(dataset), context_id=i if paradigm == "TIL" else None
            )
            buffer_trainer.train(
                train,
                test,
                batch_size=CONFIG["batch_size"],
                epochs=CONFIG["epochs"],
                lr=CONFIG["lr"],
                weight_decay=CONFIG["weight_decay"],
                regulariser=regulariser,
                early_stopping=True,
                val_freq=50,
                patience=10,
                context_id=i if paradigm == "TIL" else None
            )
            results = buffer_trainer.test(test_tasks, context_list=list(range(len(test_tasks))) if paradigm=="TIL" else [None] * len(test_tasks))
            accs = [res[1] for res in results]

            print("lower_bounds:", lower_bounds)
            diffs = [accs[i] - lower_bounds[i] for i in range(len(lower_bounds))]
            min_diff_idx = diffs.index(
                min(diffs)
            )  # The assumption is that the task closest to its boundary is the one restricting further improvements
            if results[i][1] > target_acc:
                break
            elif prev_acc is not None and results[i][1] - prev_acc < CONFIG["loosening_thresh"]:
                print("Loosening task", min_diff_idx, "bounds.")
                lower_bounds[min_diff_idx] = max(lower_bounds[min_diff_idx] - CONFIG["loosening_step"], 0.01)
                buffer_trainer.min_acc_limit = lower_bounds
            prev_acc = results[i][1]
            j += 1
        buffer_calls.append(buffer_call)

        print_colored(accs, color="green")
        accuracy_matrix.append(accs)

        lower_bounds.append(max(accs[i] - CONFIG["min_acc_increment"], 0.01))

        buffer_trainer.min_acc_limit = lower_bounds

        if buffer_trainer._last_projection is not None:
            buffer_trainer.final_certificates.append(buffer_trainer.certificates[buffer_trainer._last_projection])
        if i < len(train_tasks) - 1:
            buffer_trainer.compute_rashomon_set(test, context_id=i if paradigm == "TIL" else None)
            if len(buffer):
                buffer_trainer.add_to_buffer(buffer, task_id=i, k=CONFIG["buffer_k"])
        else:
            print("Buffer calls:", buffer_calls)
            accuracy_matrix.append(buffer_trainer.final_certificates + [0])
            columns = [f"Test Task {i}" for i in range(len(test_tasks))]
            rows = [f"Task {i}" for i in range(len(test_tasks))] + ["Certificates"]
            wandb.log(
                    {
                        "accuracy_matrix": wandb.Table(
                            data=accuracy_matrix, columns=columns, rows=rows
                        )
                    }
                )

    wandb.finish(0)
# for paradigm in ["TIL", "DIL", "CIL"]:
for paradigm in ["DIL"]:
    for (buffer_label, buffer_size) in [("small", SMALL), ("medium", MEDIUM), ("large", LARGE)]:
        for i in range(5):
            with wandb.init(project='certified-continual-learning', config=CONFIG, reinit=True, tags=["final_mnist_buffer", f"buffer_{buffer_label}", f"buffer_{paradigm.lower()}"]):
                wandb.log({'seed': i})
                config = wandb.config
                run_buffer(buffer_size, i, config, paradigm=paradigm)

Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]
[400, 400, 200, 0, 0]
[47600, 47600, 47800, 48000, 48000]


Training Epochs: 100%|█████████████████████████████████████████████████████████████| 5/5 [01:06<00:00, 13.26s/it, val_loss=0.0161, val_acc=0.9951, proj=None]


Test Results: [(0.0204, 0.9934), (0.5415, 0.7594), (0.8129, 0.6829), (3.8748, 0.5316), (4.8918, 0.2146)] (Avg: (2.0283, 0.6364))
[0.9934, 0.7594, 0.6829, 0.5316, 0.2146]
Initial acc constraint violation: -0.1478 (Positive = violated)
[[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]
[0.8433999999999999, 0.0, 0.0, 0.0, 0.0]
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.84
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.99,  Min acc soft=0.99


100%|█████████████████████████████████████████████████████████████████████████| 200/200 [00:13<00:00, 14.57it/s, size=1473.08, obj=0.238, min_soft_acc=0.791]


Final bbox:  Obj=0.24,  Size=1473.08,  Min acc hard=0.80,  Min acc soft=0.80
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['42.44', '250.22', '441.57', '624.65', '807.77', '993.72', '1140.83', '1284.20', '1397.05', '1473.08']
Checkpoint certificates: ['0.95', '0.76', '0.83', '0.83', '0.83', '0.82', '0.84', '0.82', '0.81', '0.80']
----------------------- Finished Computing Rashomon set ------------------------


...

100%|█████████████████████████████████████████████████████████████████████████| 200/200 [00:09<00:00, 22.02it/s, size=1854.25, obj=0.299, min_soft_acc=0.657]


Final bbox:  Obj=0.30,  Size=1854.25,  Min acc hard=0.72,  Min acc soft=0.71
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['44.20', '265.91', '522.31', '734.76', '938.39', '1133.17', '1320.17', '1505.13', '1679.97', '1854.25']
Checkpoint certificates: ['0.78', '0.75', '0.75', '0.75', '0.72', '0.73', '0.71', '0.72', '0.72', '0.72']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs:  40%|█████████████████████████▌                                      | 2/5 [01:03<01:35, 31.72s/it, val_loss=1.8420, val_acc=0.6564, proj=9]


Early stopping at epoch 3
Test Results: [(0.0686, 0.9861), (0.5108, 0.8935), (0.491, 0.77), (1.0488, 0.6923), (1.8407, 0.6564)] (Avg: (0.7920, 0.7997))
lower_bounds: [0.8468, 0.8219, 0.632, 0.5547]
[0.9861, 0.8935, 0.77, 0.6923, 0.6564]
Buffer calls: [0, 0, 0, 0, 4]


seed,▁
seed,4


## AGEMBufferTrainer
### Class Incremental Learning

In [ ]:
from src.trainer import AGEMBufferTrainer

In [ ]:
SEED = 0
train_tasks, _, test_tasks = get_mnist_tasks(seed=SEED, emnist=True, train_val_split_ratio=1)

model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
sizes = [4800, 4000, 4000, 3200, 0]
train_tasks, buffer_tasks = zip(
    *[create_holdout_set(dataset, holdout_size=holdout) for dataset, holdout in zip(train_tasks, sizes)]
)
print([len(task) for task in buffer_tasks])
print([len(task) for task in train_tasks])

In [ ]:
buffer = MultiTaskBuffer([])
buffer_trainer = AGEMBufferTrainer(
    model,
    memory_samples=200,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["initial_target_acc"],
    min_acc_increment=0,
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="CIL",
    seed=SEED,
    buffer=buffer,
)

MAX_BUFFER_CALLS = CONFIG["max_buffer_calls"]
target_acc = 0.65
lower_bounds = []
buffer_calls = []
for i, (train, test, buffer) in enumerate(zip(train_tasks, test_tasks, buffer_tasks)):
    buffer_trainer.train(
        train,
        test,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    results = buffer_trainer.test(test_tasks)
    accs = [res[1] for res in results]

    # If results are not satisfactory, then use buffer data to recompute rashomon set and continue training
    j = 0
    buffer_call = 0
    prev_acc = None
    while (
        j < MAX_BUFFER_CALLS
        and results[i][1] < target_acc
        and i > 0
        and not buffer_trainer.buffer.is_empty()
    ):
        buffer_call += 1
        print_colored("Using buffer to recompute LID.", color="amber")

        buffer_trainer.recall_dataset, (buffer_X, buffer_y) = buffer_trainer.buffer.consume_merge(buffer_trainer.recall_dataset)
        print("Recall dataset size:", len(buffer_trainer.recall_dataset))
        dataset = torch.utils.data.TensorDataset(buffer_X, buffer_y)
        buffer_trainer.compute_rashomon_set(
            dataset, use_outer_bbox=False, batch_size=len(dataset)
        )
        buffer_trainer.train(
            train,
            test,
            batch_size=CONFIG["batch_size"],
            epochs=CONFIG["epochs"],
            lr=CONFIG["lr"],
            weight_decay=CONFIG["weight_decay"],
            early_stopping=True,
            patience=3,
            val_freq=50
        )
        results = buffer_trainer.test(test_tasks)
        accs = [res[1] for res in results]

        print("lower_bounds:", lower_bounds)
        diffs = [accs[i] - lower_bounds[i] for i in range(len(lower_bounds))]
        min_diff_idx = diffs.index(
            min(diffs)
        )  # The assumption is that the task closest to its boundary is the one restricting further improvements
        if results[i][1] > target_acc:
            break
        elif prev_acc is not None and results[i][1] - prev_acc < CONFIG["loosening_thresh"]:
            print("Loosening task", min_diff_idx, "bounds.")
            lower_bounds[min_diff_idx] = max(lower_bounds[min_diff_idx] - CONFIG["loosening_step"], 0.01)
            buffer_trainer.min_acc_limit = lower_bounds
        prev_acc = results[i][1]
        j += 1
    buffer_calls.append(buffer_call)

    print_colored(accs, color="green")

    lower_bounds.append(max(accs[i] - CONFIG["min_acc_increment"], 0.01))

    buffer_trainer.min_acc_limit = lower_bounds

    if i < len(train_tasks) - 1:
        buffer_trainer.compute_rashomon_set(test)
        buffer_trainer.add_to_buffer(buffer, task_id=i, k=CONFIG["buffer_k"])

    else:
        print("Buffer calls:", buffer_calls)
